# 01 — Extract public data

This is the **extraction** stage of the project. Its only job is to pull the raw public data we need from the web and save a faithful copy under `../data/raw/`. No cleaning, joining, or analysis happens here — that is deliberately left to notebook `02`.

**Why separate extraction from everything else?** Web sources change, go offline, or rate-limit. By snapshotting the raw inputs to disk once, the rest of the pipeline (transform → analyse) becomes fully reproducible and can run offline. If a source breaks later, we still have the cached copy.

### What this notebook collects

| # | Variable | Source |
|---|----------|--------|
| 1 | Religiously unaffiliated share (atheist + agnostic + nothing in particular) | Pew Research Center Religious Landscape Study, per-state pages |
| 2 | Firearm death rate per 100,000 (2024, age-adjusted) | Pew analysis of CDC WONDER data |
| 3 | Adult obesity prevalence (2024) | CDC BRFSS obesity map CSV |
| 4 | Imprisonment rate per 100,000 (2023) | Bureau of Justice Statistics, *Prisoners in 2023* |
| 5 | Adult literacy / numeracy estimates | NCES PIAAC Skills Map via ArcGIS Open Data |

### Design principles used throughout

- **Defensive fetching** — every request retries with backoff and checks the HTTP status.
- **Graceful fallback** — if a live source fails, each extractor falls back to a cached CSV (or, for BJS, a curated transcription) so the notebook still produces a complete 50-state file.
- **Strict validation** — extractors assert that they end with exactly 50 states, failing loudly rather than silently producing a partial table.

## Setup

The cell below imports dependencies and resolves paths. `PROJECT_ROOT` is detected by checking whether we are running from inside the `notebooks/` folder, so the notebook works whether it is launched from the repo root or from its own directory. We also create `data/raw/` if it does not exist and define a polite `User-Agent` header that identifies the project to the servers we contact.

In [1]:
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; state-religiosity-outcomes-project/0.1; "
        "+https://example.org/research)"
    )
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw directory: {RAW_DIR}")

C:\Users\diogo\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


Project root: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project
Raw directory: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\raw


## Shared state metadata

Every source labels states differently — some use full names ("Alabama"), some use postal abbreviations ("AL"), some use slugs in URLs ("alabama"). To avoid fragile fuzzy joins later, we define one canonical reference here:

- `STATES` — abbreviation → full name (the 50 states only; no DC or territories).
- `STATE_TO_ABBR` — the reverse lookup.
- `STATE_SLUGS` — full name → URL slug, used to build Pew page URLs.

The `assert len(STATES) == 50` is a guard: if the dictionary is ever edited incorrectly, the notebook stops immediately.

In [2]:
STATES: dict[str, str] = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
}
STATE_TO_ABBR: dict[str, str] = {name: abbr for abbr, name in STATES.items()}

STATE_SLUGS: dict[str, str] = {
    state: state.lower().replace(" ", "-") for state in STATE_TO_ABBR
}

assert len(STATES) == 50

## Helper functions

This cell defines the low-level utilities shared by every extractor below. Grouping them in one place keeps each source-specific extractor short and readable.

- **`fetch_text` / `fetch_bytes`** — HTTP GET with retries, exponential-ish backoff, and `raise_for_status()`. They raise a clear `RuntimeError` after exhausting retries so failures are never silent.
- **`parse_percent`** — turns Pew's display strings (`"23%"`, `"<1%"`) into numbers. Pew rounds tiny shares to "<1%"; we encode that as `0.5`.
- **`clean_numeric`** — strips commas, percent signs, and footnote marks so values like `"1,234"` parse to floats.
- **Cache helpers (`load_cached_csv`, `load_cached_pew_religiosity`)** — read a previously saved raw file *only* if it has the expected columns (and, where relevant, exactly 50 rows). These power the fallback behaviour: if today's live fetch fails, yesterday's good snapshot is reused.

> Note: the imports and state metadata are intentionally repeated here so this cell can be executed on its own after a kernel restart without re-running earlier cells.

In [3]:
from __future__ import annotations

import html as html_lib
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; state-religiosity-outcomes-project/0.1; "
        "+https://example.org/research)"
    )
}

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_RETRIES = 3
REQUEST_BACKOFF_SECONDS = 1.5

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw directory: {RAW_DIR}")

STATES: dict[str, str] = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
}
STATE_TO_ABBR: dict[str, str] = {name: abbr for abbr, name in STATES.items()}

STATE_SLUGS: dict[str, str] = {
    state: state.lower().replace(" ", "-") for state in STATE_TO_ABBR
}

assert len(STATES) == 50


def fetch_text(url: str, timeout: int = REQUEST_TIMEOUT_SECONDS, retries: int = REQUEST_RETRIES) -> str:
    """Fetch a URL as text with retries and a defensive status check."""
    last_error: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            response.raise_for_status()
            return response.text
        except requests.RequestException as exc:
            last_error = exc
            if attempt >= retries:
                break
            wait = REQUEST_BACKOFF_SECONDS * attempt
            print(f"Request failed for {url} (attempt {attempt}/{retries}): {exc}. Retrying in {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {url}: {last_error}") from last_error


def fetch_bytes(url: str, timeout: int = 60, retries: int = REQUEST_RETRIES) -> bytes:
    """Fetch a URL as bytes with retries and a defensive status check."""
    last_error: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            response.raise_for_status()
            return response.content
        except requests.RequestException as exc:
            last_error = exc
            if attempt >= retries:
                break
            wait = REQUEST_BACKOFF_SECONDS * attempt
            print(f"Request failed for {url} (attempt {attempt}/{retries}): {exc}. Retrying in {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {url}: {last_error}") from last_error


def parse_percent(value: str) -> float | None:
    """Parse strings like '23%' or '<1%' into numeric percentages."""
    cleaned = value.strip()
    if cleaned.startswith("<"):
        # Pew rounds very small values to <1. We encode this as 0.5.
        return 0.5
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*%", cleaned)
    if match is None:
        return None
    return float(match.group(1))


def clean_numeric(value: Any) -> float | None:
    """Convert strings with commas, percents, and footnote marks into floats."""
    if pd.isna(value):
        return None
    text = str(value).strip().replace(",", "")
    text = re.sub(r"[^0-9.\-]", "", text)
    if text == "":
        return None
    return float(text)


PEW_RELG_PATH = RAW_DIR / "pew_religious_unaffiliated_by_state.csv"
PEW_REQUIRED_RELG_COLUMNS = [
    "state", "state_abbr", "atheist_pct", "agnostic_pct", "nothing_in_particular_pct",
    "religiously_unaffiliated_pct", "source_url"
]


def load_cached_pew_religiosity() -> pd.DataFrame | None:
    """Load cached Pew religiosity data to fill any partially failed state fetches."""
    if not PEW_RELG_PATH.exists():
        return None
    try:
        cached = pd.read_csv(PEW_RELG_PATH)
    except Exception as exc:
        print(f"Could not read cached Pew religiosity file: {exc}")
        return None

    missing = [c for c in PEW_REQUIRED_RELG_COLUMNS if c not in cached.columns]
    if missing:
        print(f"Cached Pew religiosity file missing columns: {missing}. Replacing from fresh extraction only.")
        return None

    return cached


FIREARM_PATH = RAW_DIR / "firearm_mortality_by_state_2024.csv"
FIREARM_REQUIRED_COLUMNS = [
    "state", "state_abbr", "firearm_death_rate_2024", "source_url"
]

OBESITY_PATH = RAW_DIR / "adult_obesity_by_state_2024.csv"
OBESITY_REQUIRED_COLUMNS = ["State", "Prevalence"]

BJS_PATH = RAW_DIR / "bjs_imprisonment_rates_by_state_2023.csv"
BJS_REQUIRED_COLUMNS = [
    "state", "state_abbr", "imprisonment_rate_2023_all_ages", "imprisonment_rate_2023_adult"
]

PIAAC_PATH = RAW_DIR / "piaac_state_literacy_numeracy.csv"
PIAAC_REQUIRED_COLUMNS = ["state", "state_abbr"]


def load_cached_csv(path: Path, required_columns: list[str], required_rows: int | None = None) -> pd.DataFrame | None:
    """Load a cached CSV when it exists and contains expected shape/columns."""
    if not path.exists():
        return None
    try:
        cached = pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read cache at {path}: {exc}")
        return None

    missing = [c for c in required_columns if c not in cached.columns]
    if missing:
        print(f"Cache at {path} is missing columns: {missing}.")
        return None

    if required_rows is not None and cached.shape[0] != required_rows:
        print(f"Cache at {path} has {cached.shape[0]} rows; expected {required_rows}. Not using as fallback.")
        return None

    return cached


def load_cached_pew_religiosity() -> pd.DataFrame | None:
    """Load cached Pew religiosity data to fill individual failed states."""
    return load_cached_csv(PEW_RELG_PATH, PEW_REQUIRED_RELG_COLUMNS)


Project root: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project
Raw directory: C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\raw


## 1. Pew Religious Landscape Study state pages

This is our **religiosity predictor**. We loop over all 50 Pew state pages and, from each, read the three components of the *religiously unaffiliated* share:

```text
Atheist + Agnostic + Nothing in particular
```

The first function below (`extract_unaffiliated_from_pew_state_page`) parses one page: it isolates the text between the "Religiously unaffiliated" heading and the next "Note:" / "Margin of error" marker, then regex-matches each component. Narrowing to that window avoids accidentally matching the same words elsewhere on the page.

The second function (`extract_pew_religiosity`) orchestrates the loop, is polite with a small delay between requests, and — crucially — backfills any individual state that failed to fetch from the cached CSV. It refuses to finish unless it has a clean set of all 50 states.

**Why measure affiliation, not belief?** The transformation notebook later computes `religiously_affiliated_pct = 100 − unaffiliated`. This is a *current, state-level proxy* for religiosity. It is not identical to Pew's older "highly religious" index (which blended prayer, attendance, belief, and importance of religion). It is a reasonable, easily reproducible stand-in — see the README for the caveat.

In [4]:
def extract_unaffiliated_from_pew_state_page(state: str, html: str) -> dict[str, float | str]:
    """Extract Pew religiously unaffiliated components from one state page.

    The Pew page renders most chart headings in plain text. We use a narrow window
    after the 'Religiously unaffiliated' section to avoid matching unrelated text.
    """
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    section_match = re.search(r"Religiously unaffiliated(?P<section>.*?)(?:Note:|Margin of error)", text, flags=re.S)
    if section_match is None:
        raise ValueError(f"Could not find unaffiliated section for {state}")

    section = section_match.group("section")

    def find_component(label: str) -> float:
        component_match = re.search(rf"{re.escape(label)}\s*\n\s*(< ?1%|[0-9]+%)", section, flags=re.I)
        if component_match is None:
            raise ValueError(f"Could not find {label} for {state}")
        value = parse_percent(component_match.group(1).replace(" ", ""))
        if value is None:
            raise ValueError(f"Could not parse {label} for {state}")
        return value

    atheist = find_component("Atheist")
    agnostic = find_component("Agnostic")
    nothing = find_component("Nothing in particular")

    return {
        "state": state,
        "state_abbr": STATE_TO_ABBR[state],
        "atheist_pct": atheist,
        "agnostic_pct": agnostic,
        "nothing_in_particular_pct": nothing,
        "religiously_unaffiliated_pct": atheist + agnostic + nothing,
        "source_url": f"https://www.pewresearch.org/religious-landscape-study/state/{STATE_SLUGS[state]}/",
    }
def extract_pew_religiosity(delay_seconds: float = 0.2) -> pd.DataFrame:
    """Download Pew state pages and extract unaffiliated percentages.

    Failures for individual states are backfilled from cache when available.
    """
    rows: list[dict[str, float | str]] = []
    failed_states: list[str] = []

    for state, slug in tqdm(STATE_SLUGS.items(), desc="Pew state pages"):
        url = f"https://www.pewresearch.org/religious-landscape-study/state/{slug}/"
        try:
            html = fetch_text(url)
            rows.append(extract_unaffiliated_from_pew_state_page(state, html))
        except Exception as exc:
            print(f"Could not fetch/parse Pew state page for {state}: {exc}")
            failed_states.append(state)
        time.sleep(delay_seconds)

    if failed_states:
        print(f"Pew religiosity extraction had {len(failed_states)} state failures; attempting cache fill.")
        cached = load_cached_pew_religiosity()
        if cached is None:
            raise RuntimeError(
                "Pew religiosity extraction had state failures and cache was unavailable. "
                f"Failed states: {failed_states}"
            )
        cached_lookup = {row["state"]: row for row in cached.to_dict("records")}
        for state in failed_states:
            fallback = cached_lookup.get(state)
            if fallback is None:
                raise RuntimeError(
                    f"Pew cache is missing failed state '{state}'. "
                    "Re-run extraction when source is reachable, or repair cache."
                )
            rows.append({
                "state": fallback["state"],
                "state_abbr": fallback["state_abbr"],
                "atheist_pct": fallback["atheist_pct"],
                "agnostic_pct": fallback["agnostic_pct"],
                "nothing_in_particular_pct": fallback["nothing_in_particular_pct"],
                "religiously_unaffiliated_pct": fallback["religiously_unaffiliated_pct"],
                "source_url": fallback["source_url"],
            })

    if len(rows) != 50:
        raise RuntimeError(f"Pew religiosity extraction produced {len(rows)} state rows; expected 50.")

    result = pd.DataFrame(rows).drop_duplicates("state").sort_values("state").reset_index(drop=True)
    if result.shape[0] != 50:
        raise RuntimeError(
            f"Pew religiosity could not recover a full 50-state set; got {result.shape[0]} after dedupe."
        )
    return result

pew_religiosity = extract_pew_religiosity()
pew_religiosity.to_csv(RAW_DIR / "pew_religious_unaffiliated_by_state.csv", index=False)
pew_religiosity.head()


Pew state pages:   0%|          | 0/50 [00:00<?, ?it/s]

,state,state_abbr,atheist_pct,agnostic_pct,nothing_in_particular_pct,religiously_unaffiliated_pct,source_url
0,Alabama,AL,3.0,2.0,18.0,23.0,https://www.pewresearch.org/religious-landscap...
1,Alaska,AK,3.0,9.0,21.0,33.0,https://www.pewresearch.org/religious-landscap...
2,Arizona,AZ,3.0,5.0,22.0,30.0,https://www.pewresearch.org/religious-landscap...
3,Arkansas,AR,2.0,4.0,12.0,18.0,https://www.pewresearch.org/religious-landscap...
4,California,CA,6.0,7.0,20.0,33.0,https://www.pewresearch.org/religious-landscap...


## 2. Firearm mortality

Our first outcome: **firearm deaths per 100,000 people in 2024, age-adjusted**, taken from a Pew short-read that republishes CDC WONDER data as an interactive chart.

The chart's numbers are not in a plain HTML `<table>` — they are embedded in a JSON `data-wp-context` attribute that the page's JavaScript reads. So the extractor uses **two strategies, in order**:

1. `_extract_firearm_rows_from_context` — find the embedded JSON, confirm its header looks like the gun-deaths table, and read the rows. This is the preferred path.
2. `_extract_firearm_rows_from_text_lines` — a fallback that scrapes the rendered text block if the JSON shape ever changes.

`_coerce_firearm_rows` then normalises the rows, drops the District of Columbia (we keep to the 50 states), and fills any genuinely missing state with `NaN` so the output is always a complete 50-row frame. As a last resort, the whole step falls back to the cached CSV.

In [5]:
PEW_GUN_DEATHS_URL = "https://www.pewresearch.org/short-reads/2026/04/28/what-the-data-says-about-gun-deaths-in-the-us/"


def _extract_firearm_rows_from_context(html: str) -> list[tuple[str, str]]:
    """Extract firearm rows from Pew `data-wp-context` attributes."""
    soup = BeautifulSoup(html, "html.parser")
    rows: list[tuple[str, str]] = []

    for tag in soup.find_all():
        for value in tag.attrs.values():
            if not isinstance(value, str) or "tableData" not in value:
                continue
            try:
                payload = json.loads(html_lib.unescape(value))
            except Exception:
                continue

            table = payload.get("tableData") if isinstance(payload, dict) else None
            if not isinstance(table, dict):
                continue
            header = table.get("header")
            if not isinstance(header, list) or len(header) < 2:
                continue

            header0 = str(header[0]).strip().lower()
            header_text = " ".join(str(col).lower() for col in header)
            if header0 != "state":
                continue
            if "gun deaths" not in header_text or "100,000" not in header_text:
                continue

            raw_rows = table.get("rows")
            if not isinstance(raw_rows, list):
                continue
            for row in raw_rows:
                if not isinstance(row, (list, tuple)) or len(row) < 2:
                    continue
                state = str(row[0]).strip()
                rate_text = str(row[1]).strip()
                if state == "District of Columbia" or state not in STATE_TO_ABBR:
                    continue
                if clean_numeric(rate_text) is None:
                    continue
                rows.append((state, rate_text))

            if rows:
                return rows
    return []


def _extract_firearm_rows_from_text_lines(html: str) -> list[tuple[str, str]]:
    """Fallback table extractor from rendered text block."""
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    lower = text.lower()
    headings = [
        "fips state gun deaths per 100,000 people in 2024",
        "state gun deaths per 100,000 people in 2024",
        "gun deaths per 100,000 people in 2024",
    ]

    start = -1
    for heading in headings:
        start = lower.find(heading)
        if start != -1:
            break
    if start == -1:
        return []

    end_markers = ["download data as .csv", "download data as csv", "share this graphic"]
    end = len(text)
    for marker in end_markers:
        marker_pos = lower.find(marker, start)
        if marker_pos != -1:
            end = min(end, marker_pos)

    table_text = text[start:end]
    pattern = re.compile(r"(?m)^\s*(?:\d{1,2})\s+([A-Za-z .&-]+?)\s+([0-9]+(?:\.[0-9]+)?)\s*$")
    rows: list[tuple[str, str]] = []
    for state, rate in pattern.findall(table_text):
        state = state.strip()
        if state == "District of Columbia" or state not in STATE_TO_ABBR:
            continue
        rows.append((state, rate))
    return rows


def _coerce_firearm_rows(rows: list[tuple[str, str]], extraction_method: str, source_url: str) -> pd.DataFrame:
    parsed_rows = []
    for state, rate_text in rows:
        cleaned_rate = clean_numeric(rate_text)
        if cleaned_rate is None:
            continue
        parsed_rows.append({
            "state": state,
            "state_abbr": STATE_TO_ABBR[state],
            "firearm_death_rate_2024": float(cleaned_rate),
            "source_url": source_url,
            "extraction_method": extraction_method,
        })

    df = pd.DataFrame(parsed_rows)
    if df.empty:
        df = pd.DataFrame(columns=[
            "state", "state_abbr", "firearm_death_rate_2024", "source_url", "extraction_method"
        ])

    for state in STATE_TO_ABBR:
        if state not in set(df["state"]):
            df = pd.concat(
                [df, pd.DataFrame([{ 
                    "state": state,
                    "state_abbr": STATE_TO_ABBR[state],
                    "firearm_death_rate_2024": float("nan"),
                    "source_url": source_url,
                    "extraction_method": "missing_state_fill",
                }])],
                ignore_index=True
            )

    return df.drop_duplicates("state").sort_values("state").reset_index(drop=True)


def extract_firearm_rates_from_pew_article(url: str = PEW_GUN_DEATHS_URL) -> pd.DataFrame:
    """Extract state firearm death rates from the Pew article chart data."""
    try:
        html = fetch_text(url)
        method = "pew_chart_table_data"
        rows = _extract_firearm_rows_from_context(html)

        if len(rows) < 48:
            fallback_rows = _extract_firearm_rows_from_text_lines(html)
            if len(fallback_rows) > len(rows):
                rows = fallback_rows
                method = "text_table_fallback"

        df = _coerce_firearm_rows(rows, extraction_method=method, source_url=url)
    except Exception as exc:
        print(f"Firearm extraction failed: {exc}")
        cached = load_cached_csv(FIREARM_PATH, FIREARM_REQUIRED_COLUMNS, required_rows=50)
        if cached is None:
            raise
        print("Using cached firearm data as fallback.")
        return cached.sort_values("state").reset_index(drop=True)

    if df.shape[0] != 50:
        raise RuntimeError(f"Firearm extraction produced {df.shape[0]} states; expected 50.")
    if df["state"].nunique() != 50:
        raise RuntimeError("Firearm extraction did not produce 50 unique states.")

    return df

firearm_rates = extract_firearm_rates_from_pew_article()
firearm_rates.to_csv(RAW_DIR / "firearm_mortality_by_state_2024.csv", index=False)
firearm_rates.head()


,state,state_abbr,firearm_death_rate_2024,source_url,extraction_method
0,Alabama,AL,23.7,https://www.pewresearch.org/short-reads/2026/0...,pew_chart_table_data
1,Alaska,AK,24.4,https://www.pewresearch.org/short-reads/2026/0...,pew_chart_table_data
2,Arizona,AZ,16.9,https://www.pewresearch.org/short-reads/2026/0...,pew_chart_table_data
3,Arkansas,AR,20.6,https://www.pewresearch.org/short-reads/2026/0...,pew_chart_table_data
4,California,CA,7.0,https://www.pewresearch.org/short-reads/2026/0...,pew_chart_table_data


## 3. Adult obesity prevalence

Our second outcome: **adult obesity prevalence in 2024** from the CDC's BRFSS obesity map.

CDC publishes this as a downloadable CSV, but the exact file URL changes each release. So `discover_cdc_obesity_csv_url` first scrapes the CDC landing page looking for an obesity CSV link; only if that discovery fails does it fall back to the last-known direct URL. `extract_obesity` downloads whichever URL it found, records the `source_url` for provenance, and falls back to the cached file on any error.

The raw CDC file includes a few non-state rows (national total, territories), which is why this file has more than 50 rows here — notebook `02` filters those out.

In [6]:
CDC_OBESITY_PAGE_URL = "https://www.cdc.gov/obesity/data-and-statistics/adult-obesity-prevalence-maps.html"
CDC_OBESITY_DIRECT_CSV_URL = "https://www.cdc.gov/obesity/media/files/2025/11/2-Obesity-by-state-in-2024.csv"


def discover_cdc_obesity_csv_url(page_url: str = CDC_OBESITY_PAGE_URL) -> str:
    """Find the first CDC obesity-by-state CSV link on the CDC page."""
    html = fetch_text(page_url)
    soup = BeautifulSoup(html, "html.parser")
    for link in soup.find_all("a", href=True):
        href = str(link["href"])
        label = link.get_text(" ").strip().lower()
        if "csv" in label and "obesity" in href.lower() and href.lower().endswith(".csv"):
            return requests.compat.urljoin(page_url, href)
    return CDC_OBESITY_DIRECT_CSV_URL


def extract_obesity() -> pd.DataFrame:
    """Download adult obesity by state CSV from CDC."""
    try:
        csv_url = discover_cdc_obesity_csv_url()
        df = pd.read_csv(csv_url)
        df["source_url"] = csv_url
        return df
    except Exception as exc:
        print(f"Obesity download/parsing failed: {exc}")
        cached = load_cached_csv(OBESITY_PATH, OBESITY_REQUIRED_COLUMNS)
        if cached is None:
            raise
        print("Using cached obesity data as fallback.")
        if "source_url" not in cached.columns:
            cached = cached.copy()
            cached["source_url"] = CDC_OBESITY_DIRECT_CSV_URL
        return cached

obesity_raw = extract_obesity()
obesity_raw.to_csv(RAW_DIR / "adult_obesity_by_state_2024.csv", index=False)
obesity_raw.head()


,State,Prevalence,95% CI,source_url
0,Alabama,38.9,"(37.1, 40.7)",https://www.cdc.gov/obesity/media/files/2025/1...
1,Alaska,34.0,"(32.0, 36.0)",https://www.cdc.gov/obesity/media/files/2025/1...
2,Arizona,33.3,"(31.4, 35.3)",https://www.cdc.gov/obesity/media/files/2025/1...
3,Arkansas,38.9,"(37.2, 40.7)",https://www.cdc.gov/obesity/media/files/2025/1...
4,California,29.1,"(27.6, 30.5)",https://www.cdc.gov/obesity/media/files/2025/1...


## 4. Bureau of Justice Statistics imprisonment rates

Our third outcome: **imprisonment rate per 100,000 residents in 2023**, from the BJS *Prisoners in 2023 – Statistical Tables* PDF (Table 7, which reports rates per 100,000 for the total population and for adults).

PDF table extraction is notoriously brittle, so this step is layered:

1. Download the PDF and try to parse Table 7 with `pdfplumber` + a regex tuned to the row layout.
2. If parsing yields fewer than 50 states (or `pdfplumber` is not installed), fall back to the cached CSV.
3. If there is no cache, use `BJS_TABLE7_FALLBACK` — a **curated, hand-transcribed copy** of the table values, clearly tagged with `extraction_method = "curated_fallback_from_bjs_table_7"` so the provenance is never ambiguous.

In the run shown below, `pdfplumber` was not available, so the curated fallback was used — which is exactly the safety net working as intended.

In [7]:
BJS_PDF_URL = "https://bjs.ojp.gov/document/p23st.pdf"

BJS_TABLE7_FALLBACK: dict[str, dict[str, int]] = {
    "Alabama": {"imprisonment_rate_2023_all_ages": 394, "imprisonment_rate_2023_adult": 505},
    "Alaska": {"imprisonment_rate_2023_all_ages": 202, "imprisonment_rate_2023_adult": 266},
    "Arizona": {"imprisonment_rate_2023_all_ages": 448, "imprisonment_rate_2023_adult": 568},
    "Arkansas": {"imprisonment_rate_2023_all_ages": 596, "imprisonment_rate_2023_adult": 773},
    "California": {"imprisonment_rate_2023_all_ages": 246, "imprisonment_rate_2023_adult": 314},
    "Colorado": {"imprisonment_rate_2023_all_ages": 290, "imprisonment_rate_2023_adult": 364},
    "Connecticut": {"imprisonment_rate_2023_all_ages": 183, "imprisonment_rate_2023_adult": 229},
    "Delaware": {"imprisonment_rate_2023_all_ages": 275, "imprisonment_rate_2023_adult": 346},
    "Florida": {"imprisonment_rate_2023_all_ages": 382, "imprisonment_rate_2023_adult": 474},
    "Georgia": {"imprisonment_rate_2023_all_ages": 449, "imprisonment_rate_2023_adult": 582},
    "Hawaii": {"imprisonment_rate_2023_all_ages": 165, "imprisonment_rate_2023_adult": 207},
    "Idaho": {"imprisonment_rate_2023_all_ages": 490, "imprisonment_rate_2023_adult": 641},
    "Illinois": {"imprisonment_rate_2023_all_ages": 238, "imprisonment_rate_2023_adult": 303},
    "Indiana": {"imprisonment_rate_2023_all_ages": 351, "imprisonment_rate_2023_adult": 455},
    "Iowa": {"imprisonment_rate_2023_all_ages": 274, "imprisonment_rate_2023_adult": 354},
    "Kansas": {"imprisonment_rate_2023_all_ages": 305, "imprisonment_rate_2023_adult": 399},
    "Kentucky": {"imprisonment_rate_2023_all_ages": 423, "imprisonment_rate_2023_adult": 545},
    "Louisiana": {"imprisonment_rate_2023_all_ages": 617, "imprisonment_rate_2023_adult": 804},
    "Maine": {"imprisonment_rate_2023_all_ages": 119, "imprisonment_rate_2023_adult": 144},
    "Maryland": {"imprisonment_rate_2023_all_ages": 261, "imprisonment_rate_2023_adult": 334},
    "Massachusetts": {"imprisonment_rate_2023_all_ages": 96, "imprisonment_rate_2023_adult": 118},
    "Michigan": {"imprisonment_rate_2023_all_ages": 328, "imprisonment_rate_2023_adult": 415},
    "Minnesota": {"imprisonment_rate_2023_all_ages": 152, "imprisonment_rate_2023_adult": 196},
    "Mississippi": {"imprisonment_rate_2023_all_ages": 652, "imprisonment_rate_2023_adult": 847},
    "Missouri": {"imprisonment_rate_2023_all_ages": 385, "imprisonment_rate_2023_adult": 494},
    "Montana": {"imprisonment_rate_2023_all_ages": 438, "imprisonment_rate_2023_adult": 552},
    "Nebraska": {"imprisonment_rate_2023_all_ages": 295, "imprisonment_rate_2023_adult": 390},
    "Nevada": {"imprisonment_rate_2023_all_ages": 327, "imprisonment_rate_2023_adult": 415},
    "New Hampshire": {"imprisonment_rate_2023_all_ages": 151, "imprisonment_rate_2023_adult": 183},
    "New Jersey": {"imprisonment_rate_2023_all_ages": 125, "imprisonment_rate_2023_adult": 160},
    "New Mexico": {"imprisonment_rate_2023_all_ages": 259, "imprisonment_rate_2023_adult": 328},
    "New York": {"imprisonment_rate_2023_all_ages": 167, "imprisonment_rate_2023_adult": 209},
    "North Carolina": {"imprisonment_rate_2023_all_ages": 275, "imprisonment_rate_2023_adult": 350},
    "North Dakota": {"imprisonment_rate_2023_all_ages": 240, "imprisonment_rate_2023_adult": 313},
    "Ohio": {"imprisonment_rate_2023_all_ages": 394, "imprisonment_rate_2023_adult": 504},
    "Oklahoma": {"imprisonment_rate_2023_all_ages": 545, "imprisonment_rate_2023_adult": 715},
    "Oregon": {"imprisonment_rate_2023_all_ages": 291, "imprisonment_rate_2023_adult": 361},
    "Pennsylvania": {"imprisonment_rate_2023_all_ages": 300, "imprisonment_rate_2023_adult": 376},
    "Rhode Island": {"imprisonment_rate_2023_all_ages": 122, "imprisonment_rate_2023_adult": 150},
    "South Carolina": {"imprisonment_rate_2023_all_ages": 299, "imprisonment_rate_2023_adult": 380},
    "South Dakota": {"imprisonment_rate_2023_all_ages": 405, "imprisonment_rate_2023_adult": 533},
    "Tennessee": {"imprisonment_rate_2023_all_ages": 341, "imprisonment_rate_2023_adult": 436},
    "Texas": {"imprisonment_rate_2023_all_ages": 477, "imprisonment_rate_2023_adult": 634},
    "Utah": {"imprisonment_rate_2023_all_ages": 186, "imprisonment_rate_2023_adult": 255},
    "Vermont": {"imprisonment_rate_2023_all_ages": 124, "imprisonment_rate_2023_adult": 150},
    "Virginia": {"imprisonment_rate_2023_all_ages": 314, "imprisonment_rate_2023_adult": 400},
    "Washington": {"imprisonment_rate_2023_all_ages": 181, "imprisonment_rate_2023_adult": 229},
    "West Virginia": {"imprisonment_rate_2023_all_ages": 328, "imprisonment_rate_2023_adult": 409},
    "Wisconsin": {"imprisonment_rate_2023_all_ages": 346, "imprisonment_rate_2023_adult": 438},
    "Wyoming": {"imprisonment_rate_2023_all_ages": 378, "imprisonment_rate_2023_adult": 484},
}


def fallback_bjs_table7() -> pd.DataFrame:
    """Return a curated transcription of BJS Table 7, with source metadata."""
    rows = []
    for state, values in BJS_TABLE7_FALLBACK.items():
        rows.append({
            "state": state,
            "state_abbr": STATE_TO_ABBR[state],
            **values,
            "source_url": BJS_PDF_URL,
            "extraction_method": "curated_fallback_from_bjs_table_7",
        })
    return pd.DataFrame(rows).sort_values("state").reset_index(drop=True)


def extract_bjs_imprisonment_rates() -> pd.DataFrame:
    """Download and parse BJS Table 7 from the PDF.

    If PDF parsing fails or returns fewer than 50 states, use cached or curated fallback.
    """
    try:
        pdf_path = RAW_DIR / "bjs_prisoners_2023_statistical_tables.pdf"
        pdf_path.write_bytes(fetch_bytes(BJS_PDF_URL))

        import pdfplumber

        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join(page.extract_text() or "" for page in pdf.pages)

        state_pattern = "|".join(re.escape(s) for s in sorted(STATE_TO_ABBR, key=len, reverse=True))
        row_pattern = re.compile(
            rf"(?m)^({state_pattern})(?:[a-z,]*)?\s+"
            r"([0-9,]+)\s+([0-9,]+)\s+([0-9,]+)\s+([0-9,]+)\s+"
            r"([0-9,]+)\s+([0-9,]+)\s+([0-9,]+)\s+([0-9,]+)\s*$"
        )

        rows = []
        for match in row_pattern.finditer(text):
            state = match.group(1)
            values = [int(x.replace(",", "")) for x in match.groups()[1:]]
            rows.append({
                "state": state,
                "state_abbr": STATE_TO_ABBR[state],
                "imprisonment_rate_2022_all_ages": values[0],
                "imprisonment_rate_2022_male": values[1],
                "imprisonment_rate_2022_female": values[2],
                "imprisonment_rate_2022_adult": values[3],
                "imprisonment_rate_2023_all_ages": values[4],
                "imprisonment_rate_2023_male": values[5],
                "imprisonment_rate_2023_female": values[6],
                "imprisonment_rate_2023_adult": values[7],
                "source_url": BJS_PDF_URL,
                "extraction_method": "pdfplumber_regex",
            })

        df = pd.DataFrame(rows).drop_duplicates("state")
        df = df[df["state"].isin(STATE_TO_ABBR)]
        if df.shape[0] == 50:
            return df.sort_values("state").reset_index(drop=True)
        print(f"PDF parser returned {df.shape[0]} states. Falling back to cached/curated BJS table.")
        raise RuntimeError("PDF parser did not return 50 states.")
    except Exception as exc:
        print(f"BJS extraction failed: {exc}")

    cached = load_cached_csv(BJS_PATH, BJS_REQUIRED_COLUMNS)
    if cached is not None and cached.shape[0] == 50:
        print("Using cached BJS data as fallback.")
        return cached.sort_values("state").reset_index(drop=True)

    print("Using curated BJS Table 7 fallback.")
    return fallback_bjs_table7()

imprisonment = extract_bjs_imprisonment_rates()
imprisonment.to_csv(RAW_DIR / "bjs_imprisonment_rates_by_state_2023.csv", index=False)
imprisonment.head()


BJS extraction failed: No module named 'pdfplumber'
Using cached BJS data as fallback.


,state,state_abbr,imprisonment_rate_2023_all_ages,imprisonment_rate_2023_adult,source_url,extraction_method
0,Alabama,AL,394,505,https://bjs.ojp.gov/document/p23st.pdf,curated_fallback_from_bjs_table_7
1,Alaska,AK,202,266,https://bjs.ojp.gov/document/p23st.pdf,curated_fallback_from_bjs_table_7
2,Arizona,AZ,448,568,https://bjs.ojp.gov/document/p23st.pdf,curated_fallback_from_bjs_table_7
3,Arkansas,AR,596,773,https://bjs.ojp.gov/document/p23st.pdf,curated_fallback_from_bjs_table_7
4,California,CA,246,314,https://bjs.ojp.gov/document/p23st.pdf,curated_fallback_from_bjs_table_7


## 5. NCES / PIAAC adult literacy and numeracy

Our fourth outcome: **adult literacy** from the NCES PIAAC Skills Map. There is no single tidy CSV for this; the data lives behind an ArcGIS FeatureServer.

The extractor therefore:

1. Searches the public ArcGIS catalogue (`search_arcgis_items`) for the PIAAC state-indicators dataset.
2. For each candidate item, queries the FeatureServer layer (`query_arcgis_feature_service`) and keeps the first response that actually contains a state column.
3. Tags the result with the source item id and URL for provenance.

Because this dataset is wide and its column names are technical (`Lit_A`, `Lit_P1`, …), we save it **as-is** here and defer choosing the right literacy column to notebook `02`. If every endpoint fails, `fallback_piaac_state_data` returns a 50-row placeholder of `NA`s so the pipeline still runs end-to-end rather than crashing.

In [8]:
ARCGIS_SEARCH_URL = "https://www.arcgis.com/sharing/rest/search"
PIAAC_SEARCH_QUERY = 'title:"PIAAC State Indicators of Adult Literacy and Numeracy"'


def search_arcgis_items(query: str, max_items: int = 10) -> list[dict[str, Any]]:
    """Search ArcGIS public catalogue and return item metadata."""
    params = {
        "f": "json",
        "q": query,
        "num": max_items,
        "sortField": "relevance",
    }
    response = requests.get(ARCGIS_SEARCH_URL, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    return list(payload.get("results", []))


def query_arcgis_feature_service(service_url: str) -> pd.DataFrame:
    """Query all records from an ArcGIS FeatureServer layer."""
    query_url = service_url.rstrip("/") + "/query"
    params = {
        "f": "json",
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "resultRecordCount": 10000,
    }
    response = requests.get(query_url, params=params, headers=HEADERS, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise ValueError(payload["error"])
    features = payload.get("features", [])
    return pd.DataFrame([feature.get("attributes", {}) for feature in features])


def _candidate_arcgis_urls(value: str) -> list[str]:
    candidates: list[str] = []
    if not value:
        return candidates
    normalized = value.rstrip("/")
    if "FeatureServer" in normalized or "MapServer" in normalized:
        if not normalized.endswith("/0"):
            candidates.append(normalized + "/0")
        else:
            candidates.append(normalized)
    else:
        candidates.append(normalized)
    return candidates


def fallback_piaac_state_data() -> pd.DataFrame:
    """Fallback placeholder that keeps the downstream transform pipeline robust."""
    rows = []
    for state in sorted(STATE_TO_ABBR):
        rows.append({
            "state": state,
            "state_abbr": STATE_TO_ABBR[state],
            "literacy_avg_score": pd.NA,
            "low_literacy_pct": pd.NA,
            "source_url": PIAAC_SEARCH_QUERY,
            "source_id": "piaac_fallback",
            "source_title": "PIAAC fallback",
            "extraction_method": "curated_fallback_empty",
        })
    return pd.DataFrame(rows)


def extract_piaac_state_data() -> pd.DataFrame:
    """Find and extract the PIAAC state-level literacy/numeracy data."""
    try:
        items = search_arcgis_items(PIAAC_SEARCH_QUERY, max_items=25)
        if not items:
            print("No ArcGIS items found for PIAAC state indicators.")
            raise RuntimeError("No ArcGIS items found.")

        candidates = [
            item for item in items
            if item.get("url") and any(token in item.get("url", "") for token in ["FeatureServer", "MapServer"])
        ]
        if not candidates:
            candidates = [item for item in items if item.get("url")]
        if not candidates:
            print("No usable ArcGIS service URL for PIAAC.")
            raise RuntimeError("No usable ArcGIS service URL.")

        for item in candidates:
            for service_url in _candidate_arcgis_urls(item.get("url", "")):
                try:
                    df = query_arcgis_feature_service(service_url)
                except Exception as exc:
                    print(f"Failed to query ArcGIS service {service_url}: {exc}. Trying another source.")
                    continue

                if df.empty:
                    continue
                if "state" not in [c.lower() for c in df.columns] and "State" not in df.columns:
                    continue

                df["source_item_id"] = item.get("id")
                df["source_title"] = item.get("title")
                df["source_url"] = service_url
                df["extraction_method"] = "arcgis_feature_service"
                return df

        print("All ArcGIS PIAAC service endpoints failed.")
        raise RuntimeError("ArcGIS endpoints failed for PIAAC.")
    except Exception as exc:
        print(f"PIAAC online extraction failed: {exc}")
        cached = load_cached_csv(PIAAC_PATH, PIAAC_REQUIRED_COLUMNS, required_rows=50)
        if cached is not None:
            print("Using cached PIAAC data as fallback.")
            return cached
        # Do not clobber a good existing raw file with an empty placeholder:
        # the real ArcGIS export uses a "State" column and "Lit_*" fields.
        if PIAAC_PATH.exists():
            existing = pd.read_csv(PIAAC_PATH)
            if any("lit" in c.lower() for c in existing.columns):
                print("Live fetch failed; keeping existing PIAAC raw file with real literacy data.")
                return existing
        print("Using empty PIAAC fallback placeholder.")
        return fallback_piaac_state_data()

piaac_state = extract_piaac_state_data()
piaac_state.to_csv(RAW_DIR / "piaac_state_literacy_numeracy.csv", index=False)
piaac_state.head()


Failed to query ArcGIS service https://nces.ed.gov/opengis/rest/services/Social_Economic/NCES_PIAAC_SKILLSMAP_STATE_2017/MapServer/0: 500 Server Error: Internal Server Error for url: https://nces.ed.gov/opengis/rest/services/Social_Economic/NCES_PIAAC_SKILLSMAP_STATE_2017/MapServer/0/query?f=json&where=1%3D1&outFields=%2A&returnGeometry=false&resultRecordCount=10000. Trying another source.


Failed to query ArcGIS service https://nces.ed.gov/opengis/rest/services/Social_Economic/NCES_PIAAC_SKILLSMAP_STATE_2017/MapServer/0: 500 Server Error: Internal Server Error for url: https://nces.ed.gov/opengis/rest/services/Social_Economic/NCES_PIAAC_SKILLSMAP_STATE_2017/MapServer/0/query?f=json&where=1%3D1&outFields=%2A&returnGeometry=false&resultRecordCount=10000. Trying another source.
All ArcGIS PIAAC service endpoints failed.
PIAAC online extraction failed: ArcGIS endpoints failed for PIAAC.
Cache at C:\Users\diogo\work_code\temporary\religiosity_us_outcomes_project\data\raw\piaac_state_literacy_numeracy.csv is missing columns: ['state', 'state_abbr'].
Live fetch failed; keeping existing PIAAC raw file with real literacy data.


,State,Lit_P1,Lit_P1_CI_L,Lit_P1_CI_U,Lit_P1_CV,Lit_P1_CI_L_indicator,Lit_P2,Lit_P2_CI_L,Lit_P2_CI_U,Lit_P2_CV,...,No_Insurance,Shape_Length,Shape_Area,OBJECTID_1,STATEFP,STUSPS,source_item_id,source_title,source_url,extraction_method
0,Alabama,0.239,0.209,0.270,0.063,NaN,0.367,0.324,0.412,0.058,...,0.107,2.258425e+06,1.902196e+11,1,1,AL,c5551d52a6484c83a872f9944a881a6d,PIAAC State Indicators of Adult Literacy and N...,https://nces.ed.gov/opengis/rest/services/Soci...,arcgis_feature_service
1,Alaska,0.127,0.092,0.162,0.138,NaN,0.329,0.274,0.381,0.082,...,0.155,5.836330e+07,8.232612e+12,2,2,AK,c5551d52a6484c83a872f9944a881a6d,PIAAC State Indicators of Adult Literacy and N...,https://nces.ed.gov/opengis/rest/services/Soci...,arcgis_feature_service
2,Arizona,0.234,0.199,0.265,0.070,NaN,0.316,0.272,0.368,0.078,...,0.122,2.871255e+06,4.341164e+11,3,4,AZ,c5551d52a6484c83a872f9944a881a6d,PIAAC State Indicators of Adult Literacy and N...,https://nces.ed.gov/opengis/rest/services/Soci...,arcgis_feature_service
3,Colorado,0.166,0.138,0.192,0.081,NaN,0.297,0.257,0.348,0.078,...,0.094,2.707563e+06,4.473030e+11,4,8,CO,c5551d52a6484c83a872f9944a881a6d,PIAAC State Indicators of Adult Literacy and N...,https://nces.ed.gov/opengis/rest/services/Soci...,arcgis_feature_service
4,Florida,0.237,0.211,0.263,0.056,NaN,0.343,0.309,0.376,0.050,...,0.149,4.365387e+06,1.977060e+11,5,12,FL,c5551d52a6484c83a872f9944a881a6d,PIAAC State Indicators of Adult Literacy and N...,https://nces.ed.gov/opengis/rest/services/Soci...,arcgis_feature_service


## 6. CDC firearm deaths by intent (+ a gun-ownership proxy)

The total firearm rate from Pew hides a crucial distinction: **firearm homicide** and **firearm suicide** behave very differently across states (rural, high-ownership states tend to be low on homicide but very high on suicide). We pull both from the CDC's *Mapping Injury, Overdose, and Violence* dataset (Socrata `fpsi-y8tj`), period 2024:

- `firearm_homicide_rate_2024`, `firearm_suicide_rate_2024` (per 100,000),
- `firearm_total_rate_cdc_2024` (CDC's all-intent firearm rate, for cross-check).

From the same data we also build a **household gun-ownership proxy**: the FS/S ratio (firearm suicides ÷ all suicides). This is a standard, validated proxy used in the public-health literature when direct ownership surveys are unavailable — and gun ownership is the single most important confounder for firearm outcomes.

In [9]:
import urllib.parse

CDC_FIREARM_INTENT_URL = "https://data.cdc.gov/resource/fpsi-y8tj.json"
CDC_FIREARM_PERIOD = "2024"
CDC_FIREARM_PATH = RAW_DIR / "cdc_firearm_intent_by_state_2024.csv"
CDC_FIREARM_REQUIRED_COLUMNS = [
    "state", "state_abbr", "firearm_homicide_rate_2024",
    "firearm_suicide_rate_2024", "gun_ownership_fss_proxy",
]


def _fetch_cdc_intent(intent: str, period: str = CDC_FIREARM_PERIOD) -> list:
    query = urllib.parse.urlencode({"intent": intent, "period": period, "$limit": 500})
    return json.loads(fetch_text(f"{CDC_FIREARM_INTENT_URL}?{query}"))


def extract_cdc_firearm_intent() -> pd.DataFrame:
    """Firearm deaths split by intent + an FS/S household gun-ownership proxy."""
    try:
        records: dict[str, dict] = {}
        for intent in ["FA_Deaths", "FA_Homicide", "FA_Suicide", "All_Suicide"]:
            for row in _fetch_cdc_intent(intent):
                name = str(row.get("name", "")).strip()
                if name not in STATE_TO_ABBR:  # drops DC, United States, territories
                    continue
                rec = records.setdefault(name, {})
                rec[f"{intent}_rate"] = clean_numeric(row.get("rate"))
                rec[f"{intent}_count"] = clean_numeric(row.get("count_sup"))

        rows = []
        for state, rec in records.items():
            fa_sui = rec.get("FA_Suicide_count")
            all_sui = rec.get("All_Suicide_count")
            fss = (fa_sui / all_sui) if (fa_sui is not None and all_sui) else None
            rows.append({
                "state": state,
                "state_abbr": STATE_TO_ABBR[state],
                "firearm_total_rate_cdc_2024": rec.get("FA_Deaths_rate"),
                "firearm_homicide_rate_2024": rec.get("FA_Homicide_rate"),
                "firearm_suicide_rate_2024": rec.get("FA_Suicide_rate"),
                "gun_ownership_fss_proxy": fss,
                "source_url": f"{CDC_FIREARM_INTENT_URL} (intent split, period {CDC_FIREARM_PERIOD})",
                "extraction_method": "cdc_socrata_fpsi_y8tj",
            })
        df = pd.DataFrame(rows).drop_duplicates("state").sort_values("state").reset_index(drop=True)
        if df.shape[0] != 50:
            raise RuntimeError(f"CDC firearm intent produced {df.shape[0]} states; expected 50.")
        return df
    except Exception as exc:
        print(f"CDC firearm intent extraction failed: {exc}")
        cached = load_cached_csv(CDC_FIREARM_PATH, CDC_FIREARM_REQUIRED_COLUMNS, required_rows=50)
        if cached is not None:
            print("Using cached CDC firearm intent data.")
            return cached.sort_values("state").reset_index(drop=True)
        raise

cdc_firearm = extract_cdc_firearm_intent()
cdc_firearm.to_csv(CDC_FIREARM_PATH, index=False)
cdc_firearm.head()


,state,state_abbr,firearm_total_rate_cdc_2024,firearm_homicide_rate_2024,firearm_suicide_rate_2024,gun_ownership_fss_proxy,source_url,extraction_method
0,Alabama,AL,23.7,11.0,12.2,0.743130,https://data.cdc.gov/resource/fpsi-y8tj.json (...,cdc_socrata_fpsi_y8tj
1,Alaska,AK,24.8,4.5,19.2,0.635135,https://data.cdc.gov/resource/fpsi-y8tj.json (...,cdc_socrata_fpsi_y8tj
2,Arizona,AZ,17.9,4.4,12.6,0.615435,https://data.cdc.gov/resource/fpsi-y8tj.json (...,cdc_socrata_fpsi_y8tj
3,Arkansas,AR,20.8,6.9,13.1,0.679054,https://data.cdc.gov/resource/fpsi-y8tj.json (...,cdc_socrata_fpsi_y8tj
4,California,CA,7.3,3.1,3.9,0.380408,https://data.cdc.gov/resource/fpsi-y8tj.json (...,cdc_socrata_fpsi_y8tj


## 7. State land area (for an urbanicity / population-density control)

Urbanisation is a classic confounder. We do not have a direct "% urban" feed, but we can build a solid **population-density** proxy: state population (already in the PIAAC export) divided by land area. Land area comes from the Census county **Gazetteer** file (keyless), aggregated to the state level. Notebook 02 turns this into a `population_density` covariate. We also compute an **area-weighted state centroid** (lat/long) here, used later for the spatial-autocorrelation check.

In [10]:
import io
import zipfile

CENSUS_GAZETTEER_URL = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2023_Gazetteer/2023_Gaz_counties_national.zip"
LAND_AREA_PATH = RAW_DIR / "census_state_land_area.csv"
LAND_AREA_REQUIRED_COLUMNS = ["state", "state_abbr", "land_area_sqmi", "centroid_lat", "centroid_lon"]


def extract_state_land_area() -> pd.DataFrame:
    """State land area (sq mi) + an area-weighted centroid, from the county Gazetteer."""
    try:
        raw = fetch_bytes(CENSUS_GAZETTEER_URL)
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            member = next(n for n in zf.namelist() if n.lower().endswith(".txt"))
            with zf.open(member) as fh:
                gaz = pd.read_csv(fh, sep="\t", dtype=str, encoding="latin-1")
        gaz.columns = [c.strip() for c in gaz.columns]
        lat_col = next(c for c in gaz.columns if c.upper() == "INTPTLAT")
        lon_col = next(c for c in gaz.columns if c.upper() in ("INTPTLONG", "INTPTLON"))
        gaz["ALAND_SQMI"] = gaz["ALAND_SQMI"].map(clean_numeric)
        gaz["_lat"] = gaz[lat_col].map(clean_numeric)
        gaz["_lon"] = gaz[lon_col].map(clean_numeric)
        gaz = gaz[gaz["USPS"].isin(STATES.keys())].dropna(subset=["ALAND_SQMI", "_lat", "_lon"]).copy()
        rows = []
        for usps, g in gaz.groupby("USPS"):
            wsum = g["ALAND_SQMI"].sum()
            rows.append({
                "state": STATES[usps],
                "state_abbr": usps,
                "land_area_sqmi": float(g["ALAND_SQMI"].sum()),
                "centroid_lat": float((g["ALAND_SQMI"] * g["_lat"]).sum() / wsum),
                "centroid_lon": float((g["ALAND_SQMI"] * g["_lon"]).sum() / wsum),
                "source_url": CENSUS_GAZETTEER_URL,
            })
        agg = pd.DataFrame(rows).sort_values("state").reset_index(drop=True)
        if agg.shape[0] != 50:
            raise RuntimeError(f"Land area produced {agg.shape[0]} states; expected 50.")
        return agg
    except Exception as exc:
        print(f"State land area extraction failed: {exc}")
        cached = load_cached_csv(LAND_AREA_PATH, LAND_AREA_REQUIRED_COLUMNS, required_rows=50)
        if cached is not None:
            print("Using cached land area data.")
            return cached.sort_values("state").reset_index(drop=True)
        raise

state_land_area = extract_state_land_area()
state_land_area.to_csv(LAND_AREA_PATH, index=False)
state_land_area.head()


,state,state_abbr,land_area_sqmi,centroid_lat,centroid_lon,source_url
0,Alabama,AL,50650.834,32.759800,-86.830226,https://www2.census.gov/geo/docs/maps-data/dat...
1,Alaska,AK,571051.641,63.726169,-149.836218,https://www2.census.gov/geo/docs/maps-data/dat...
2,Arizona,AZ,113655.393,34.269321,-111.666668,https://www2.census.gov/geo/docs/maps-data/dat...
3,Arkansas,AR,51992.701,34.891137,-92.442263,https://www2.census.gov/geo/docs/maps-data/dat...
4,California,CA,155859.140,37.147771,-119.546925,https://www2.census.gov/geo/docs/maps-data/dat...


## 8. Census ACS: median household income and median age

Two more classic confounders from the Census ACS 1-year (2023) API: **median household income** (`B19013_001E`) and **median age** (`B01002_001E`) by state.

This needs a free [Census API key](https://api.census.gov/data/key_signup.html) supplied via the `CENSUS_API_KEY` environment variable — the key is **never** written into the notebook. If the key is missing or the request fails, the extractor falls back to the cached CSV (non-destructive), so the pipeline still runs without it.

In [11]:
import os

CENSUS_ACS_URL = "https://api.census.gov/data/2023/acs/acs1"
CENSUS_VARS = {"B19013_001E": "median_household_income", "B01002_001E": "median_age"}
CENSUS_INCOME_AGE_PATH = RAW_DIR / "census_income_age_by_state.csv"
CENSUS_INCOME_AGE_REQUIRED_COLUMNS = ["state", "state_abbr", "median_household_income", "median_age"]


def extract_census_income_age() -> pd.DataFrame:
    """Median household income + median age by state (Census ACS 1-year API).

    Reads the key from CENSUS_API_KEY. The key is passed via request params, never
    logged or printed, and never stored in the notebook. Falls back to cache otherwise.
    """
    try:
        key = os.environ.get("CENSUS_API_KEY")
        if not key:
            raise RuntimeError("CENSUS_API_KEY not set in environment.")
        params = {"get": "NAME," + ",".join(CENSUS_VARS), "for": "state:*", "key": key}
        resp = requests.get(CENSUS_ACS_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
        resp.raise_for_status()
        header, *data = resp.json()
        idx = {name: i for i, name in enumerate(header)}
        rows = []
        for rec in data:
            name = rec[idx["NAME"]].strip()
            if name not in STATE_TO_ABBR:  # drops DC and Puerto Rico
                continue
            rows.append({
                "state": name,
                "state_abbr": STATE_TO_ABBR[name],
                "median_household_income": clean_numeric(rec[idx["B19013_001E"]]),
                "median_age": clean_numeric(rec[idx["B01002_001E"]]),
                "source_url": CENSUS_ACS_URL + " (ACS 1-year 2023)",
                "extraction_method": "census_acs_api",
            })
        df = pd.DataFrame(rows).drop_duplicates("state").sort_values("state").reset_index(drop=True)
        if df.shape[0] != 50:
            raise RuntimeError(f"Census ACS produced {df.shape[0]} states; expected 50.")
        return df
    except Exception as exc:
        # Print only the exception type so an API key embedded in a URL can never leak.
        print(f"Census ACS income/age extraction failed ({type(exc).__name__}); trying cache.")
        cached = load_cached_csv(CENSUS_INCOME_AGE_PATH, CENSUS_INCOME_AGE_REQUIRED_COLUMNS, required_rows=50)
        if cached is not None:
            print("Using cached Census income/age data.")
            return cached.sort_values("state").reset_index(drop=True)
        raise

census_income_age = extract_census_income_age()
census_income_age.to_csv(CENSUS_INCOME_AGE_PATH, index=False)
census_income_age.head()


,state,state_abbr,median_household_income,median_age,source_url,extraction_method
0,Alabama,AL,62212.0,39.6,https://api.census.gov/data/2023/acs/acs1 (ACS...,census_acs_api
1,Alaska,AK,86631.0,36.5,https://api.census.gov/data/2023/acs/acs1 (ACS...,census_acs_api
2,Arizona,AZ,77315.0,39.3,https://api.census.gov/data/2023/acs/acs1 (ACS...,census_acs_api
3,Arkansas,AR,58700.0,38.9,https://api.census.gov/data/2023/acs/acs1 (ACS...,census_acs_api
4,California,CA,95521.0,38.2,https://api.census.gov/data/2023/acs/acs1 (ACS...,census_acs_api


## Extraction summary

A final sanity check: list every CSV now sitting in `data/raw/` with its shape. We expect roughly 50 rows per file (obesity and PIAAC carry a few extra non-state rows that notebook `02` will drop). If any file is missing or unexpectedly small, that is the signal to re-run the relevant extractor before moving on.

**Hand-off:** with the raw snapshots saved, the next notebook (`02_transform_state_panel`) can clean, standardise, and merge these files into a single analytical table — without touching the network again.

In [12]:
for path in sorted(RAW_DIR.glob("*.csv")):
    df = pd.read_csv(path)
    print(f"{path.name}: {df.shape[0]} rows, {df.shape[1]} columns")

adult_obesity_by_state_2024.csv: 54 rows, 4 columns
bjs_imprisonment_rates_by_state_2023.csv: 50 rows, 6 columns
cdc_firearm_intent_by_state_2024.csv: 50 rows, 8 columns
census_income_age_by_state.csv: 50 rows, 6 columns
census_state_land_area.csv: 50 rows, 6 columns
firearm_mortality_by_state_2024.csv: 50 rows, 5 columns
pew_religious_unaffiliated_by_state.csv: 50 rows, 7 columns
piaac_state_literacy_numeracy.csv: 51 rows, 79 columns
